In [1]:
# !pip install -q -U google-genai
# !pip install python-docx
# !pip install python-dotenv

# Detect Policy Changes

In [2]:
import os
from bs4 import BeautifulSoup
import requests
import time
import re
from datetime import datetime

import docx
from docx.shared import Pt, Inches

from google import genai
from google.genai import types

from itertools import zip_longest

from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(dotenv_path="local.env")

True

In [3]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Security Error: GEMINI_API_KEY environment variable is missing!")

In [4]:
# 1. Target URL config
TARGET_URL = "https://www.legislation.wa.gov.au/legislation/statutes.nsf/law_a517_currencies.html&view=consolidated"
BASE_URL = "https://www.legislation.wa.gov.au/legislation/statutes.nsf/"

# Create a local directory for storage
output_dir = "mining_act_html_versions"
os.makedirs(output_dir, exist_ok=True)

## Fetching Mining Act 1978

In [5]:
print(f"Fetching page content from: {TARGET_URL}...")

try:
    # 2. Download the main versions index page
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    page_response = requests.get(TARGET_URL, headers=headers, timeout=15)
    page_response.raise_for_status()
    html_content = page_response.text

except Exception as e:
    print(f"Failed to fetch index page: {e}")
    exit()

soup = BeautifulSoup(html_content, "html.parser")

table = soup.find("table", class_="range compare")
if not table:
    print("Could not find the target legislation table on the live page.")
    exit()

rows = table.find("tbody").find_all("tr")
download_count = 0
skip_count = 0

print("\nStarting automated download of HTML act versions...")
print("-" * 60)

for row in rows:
    cells = row.find_all("td")
    if len(cells) < 4:
        continue

    # Extract clean naming tags from metadata columns
    act_name = cells[0].text.strip()            
    start_date = cells[1].text.strip().replace(" ", "_")  
    suffix = cells[3].text.strip()         

    # Filter out anchors that specifically direct to targeted document queries containing .htm
    html_link_node = row.find("a", href=lambda href: href and "query=" in href and ".htm" in href)

    if html_link_node:
        relative_href = html_link_node["href"]
        # Standardize matching URLs against the site's <base href> tag
        full_download_url = BASE_URL + relative_href

        # Construct safe, distinguishable file titles
        filename = f"{act_name}_{suffix}_{start_date}.html".replace("/", "-")
        filepath = os.path.join(output_dir, filename)

        # CHECK: If the file already exists in output_dir, skip the download
        if os.path.exists(filepath):
            print(f"Skipping [{suffix}] ({cells[1].text.strip()}) -> Already exists locally.")
            skip_count += 1
            continue

        print(f"Downloading version tag [{suffix}] ({cells[1].text.strip()})...")

        try:
            # Request target via the site redirect framework
            file_response = requests.get(full_download_url, headers=headers, timeout=15)
            if file_response.status_code == 200:
                with open(filepath, "w", encoding="utf-8") as file:
                    file.write(file_response.text)
                download_count += 1
                print(f"  -> Saved successfully as: {filename}")
            else:
                print(f"  -> Download failed with HTTP Status: {file_response.status_code}")
        except Exception as e:
            print(f"  -> Connection error handling filename {filename}: {e}")

print("-" * 60)
print(f"Execution finished.")
print(f"Total files downloaded: {download_count}")
print(f"Total files skipped (already exist): {skip_count}")
print(f"Files are located in the '{output_dir}' directory.")

Fetching page content from: https://www.legislation.wa.gov.au/legislation/statutes.nsf/law_a517_currencies.html&view=consolidated...

Starting automated download of HTML act versions...
------------------------------------------------------------
Skipping [09-q0-00] (28 May 2026) -> Already exists locally.
Skipping [09-p0-00] (9 Sep 2025) -> Already exists locally.
Skipping [09-o0-00] (1 Sep 2025) -> Already exists locally.
Skipping [09-n0-00] (31 Aug 2025) -> Already exists locally.
Skipping [09-m0-00] (2 Jul 2025) -> Already exists locally.
Skipping [09-l0-00] (14 May 2024) -> Already exists locally.
Skipping [09-k0-01] (18 Nov 2023) -> Already exists locally.
Skipping [09-j0-01] (10 Aug 2023) -> Already exists locally.
Skipping [09-i0-00] (5 Apr 2023) -> Already exists locally.
Skipping [09-h0-00] (24 Mar 2023) -> Already exists locally.
Skipping [09-g0-00] (2 Nov 2022) -> Already exists locally.
Skipping [09-f0-00] (28 Sep 2022) -> Already exists locally.
Skipping [09-e0-00] (1 Jul

In [6]:
def get_file_metadata(file_path):
    """
    Parses a downloaded HTML file to extract its official currency start date 
    and version suffix from the text metadata.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            # Read just the first 5000 characters to find titles/metadata quickly
            head_content = f.read(5000)
            
        # 1. Fallback to extracting metadata directly from the filename if matching a pattern
        # Format assumed: "Mining Act 1978_Suffix_Date.html"
        filename = os.path.basename(file_path)
        match = re.search(r"Mining Act 1978_(.+?)_(.+?)\.html", filename)
        
        if match:
            suffix = match.group(1)
            date_str = match.group(2).replace("_", " ")
            # Handle "Current" or specific dates safely
            parsed_date = datetime.strptime(date_str, "%d %b %Y")
            return {"path": file_path, "date": parsed_date, "suffix": suffix}
            
    except Exception as e:
        print(f"Could not parse metadata for {os.path.basename(file_path)}: {e}")
    return None

def find_latest_versions(directory):
    """
    Scans the directory, gathers version metadata, and sorts files 
    chronologically to isolate the two newest records.
    """
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        return None, None

    all_versions = []
    
    # Loop through all downloaded files in the target directory
    for filename in os.listdir(directory):
        if filename.startswith("Mining Act 1978") and filename.endswith(".html"):
            file_path = os.path.join(directory, filename)
            metadata = get_file_metadata(file_path)
            if metadata:
                all_versions.append(metadata)
                
    if len(all_versions) < 2:
        print(f"Found {len(all_versions)} file(s). You need at least 2 files to compare.")
        return None, None

    # Sort files chronologically by their parsed currency date (newest first)
    all_versions.sort(key=lambda x: x["date"], reverse=True)
    
    # Isolate the two topmost records
    newest = all_versions[0]["path"]
    second_newest = all_versions[1]["path"]
    
    return newest, second_newest

In [7]:
def extract_clean_text(file_path):
    print(f"Cleaning raw markup from: {os.path.basename(file_path)}...")
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    
    content_area = soup.find("section", id="main") or soup.find("section", id="outer")
    target_soup = content_area if content_area else soup
    for element in target_soup(["script", "style", "meta", "link", "header", "footer", "nav", "form"]):
        element.decompose()
    text = target_soup.get_text(separator="\n")
    
    cleaned_lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(cleaned_lines)

def split_into_chunks(text, chunk_size=100000):
    """
    Splits text by lines to avoid cutting a section or sentence in half.
    Targeting ~30,000 characters per chunk keeps total input tokens well under 10,000.
    """
    chunks = []
    current_chunk = []
    current_size = 0
    
    for line in text.splitlines():
        current_chunk.append(line)
        current_size += len(line)
        if current_size >= chunk_size:
            chunks.append("\n".join(current_chunk))
            current_chunk = []
            current_size = 0
            
    if current_chunk:
        chunks.append("\n".join(current_chunk))
    return chunks

def extract_date_from_filename(filename):
    """
    Helper function to pull the clean date string from the filename 
    e.g., 'Mining Act 1978_09-p0-00_9_Sep_2025.html' -> '9_Sep_2025'
    """
    match = re.search(r"Mining Act 1978_.+?_(.+?)\.html", os.path.basename(filename))
    return match.group(1) if match else "UnknownDate"

def save_markdown_to_docx(markdown_text, output_filename):
    doc = docx.Document()
    
    # Configure base style constraints
    style = doc.styles['Normal']
    font = style.font
    font.name = 'Arial'
    font.size = Pt(11)
    
    print(f"Generating formatted report configuration...")
    lines = markdown_text.split('\n')
    
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
            
        if stripped.startswith('# '):
            doc.add_heading(stripped[2:], level=1)
        elif stripped.startswith('## '):
            doc.add_heading(stripped[3:], level=2)
        elif stripped.startswith('### '):
            doc.add_heading(stripped[4:], level=3)
        elif stripped.startswith('* ') or stripped.startswith('- '):
            cleaned_bullet = re.sub(r'\*\*(.*?)\*\*', r'\1', stripped[2:])
            doc.add_paragraph(cleaned_bullet, style='List Bullet')
        else:
            cleaned_text = re.sub(r'\*\*(.*?)\*\*', r'\1', line)
            doc.add_paragraph(cleaned_text)
            
    doc.save(output_filename)
    print(f"Success! Document created at: {os.path.abspath(output_filename)}")

In [8]:
# new_mining_act_file = "Mining Act 1978_09-p0-00_9_Sep_2025.html" 
# old_mining_act_file = "Mining Act 1978_09-o0-00_1_Sep_2025.html"


OUTPUT_DIR = "mining_act_html_versions"

model= "gemma-4-31b-it"

max_retries = 5

new_mining_act_file, old_mining_act_file = find_latest_versions(OUTPUT_DIR)

print(f"New: {new_mining_act_file}")
print(f"Old: {old_mining_act_file}")
old_mining_act_file

New: mining_act_html_versions/Mining Act 1978_09-q0-00_28_May_2026.html
Old: mining_act_html_versions/Mining Act 1978_09-p0-00_9_Sep_2025.html


'mining_act_html_versions/Mining Act 1978_09-p0-00_9_Sep_2025.html'

In [9]:
# new_file_path = os.path.join(OUTPUT_DIR, new_mining_act_file)
# old_file_path = os.path.join(OUTPUT_DIR, old_mining_act_file)

old_text_content = extract_clean_text(old_mining_act_file)
new_text_content = extract_clean_text(new_mining_act_file)

print("\nChunking text into manageable segments...")
old_chunks = split_into_chunks(old_text_content)
new_chunks = split_into_chunks(new_text_content)
if not os.path.exists(new_mining_act_file) or not os.path.exists(old_mining_act_file):
    print("Error: One or both of the specified Mining Act HTML files are missing.")
    exit()

print("Reading HTML source files...")
with open(old_mining_act_file, "r", encoding="utf-8") as f:
    old_html_content = f.read()

with open(new_mining_act_file, "r", encoding="utf-8") as f:
    new_html_content = f.read()

Cleaning raw markup from: Mining Act 1978_09-p0-00_9_Sep_2025.html...
Cleaning raw markup from: Mining Act 1978_09-q0-00_28_May_2026.html...

Chunking text into manageable segments...
Reading HTML source files...


In [10]:
client = genai.Client(api_key = GEMINI_API_KEY)

In [11]:
print(f"\nStarting analysis across {max(len(old_chunks), len(new_chunks))} structural blocks...")
print("-" * 60)


Starting analysis across 6 structural blocks...
------------------------------------------------------------


In [12]:
respone_text = """"""

for idx, (old_chunk, new_chunk) in enumerate(zip_longest(old_chunks, new_chunks, fillvalue="")):
    # Base Case: Both chunks are completely identical, save your API quota!
    if old_chunk == new_chunk:
        print(f"Block {idx + 1}: No text variations found. Skipping API call.")
        continue
        
    print(f"Block {idx + 1}: Text variation detected! Requesting localized analysis...")
    
    prompt = f"""
    You are an expert legislative data analyst. 
    Compare the following text block from the older version of the Mining Act 1978 against the matching block from the newer version.

    # Please provide a structured report detailing:
    # 1. **Executive Summary**: A brief overview of how extensive the changes are.
    # 2. **Specific Section Amendments**: Detail which sections, parts, or schedules were added, deleted, or altered. 
    # 3. **Textual Differences**: Highlight significant changes in phrasing, definitions, or penalties.

    ---
    [OLD REVISION TEXT SEGMENT]
    {old_chunk}
    ---
    [NEW REVISION TEXT SEGMENT]
    {new_chunk}
    ---
    """
    
    try:
        # Generate localized diff report
        response = client.models.generate_content(
            model='gemma-4-31b-it',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
            )
        )
        
        # Display the findings for this block
        print(f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---")
        print(response.text)
        respone_text += f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---"
        respone_text += response.text
        print("-" * 60)
        
        # Pause briefly to prevent hitting requests-per-minute limits
        time.sleep(120)
        
    except Exception as e:
        max_retries = 3
        retry_delay = 120
        success = False
        
        for attempt in range(max_retries):
            try:
                time.sleep(120)
                print(f"Retrying in 120 seconds...")
                # Generate localized diff report
                response = client.models.generate_content(
                    model='gemma-4-31b-it',
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.0,
                    )
                )
                
                # Display the findings for this block
                print(f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---")
                print(response.text)
                respone_text += f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---"
                respone_text += response.text
                print("-" * 60)
                
                success = True
                time.sleep(120)
                break  # Exit the retry loop on success
                
            except Exception as e:
                # Check if it's a server error (500, 503, etc.)
                print(f"Attempt {attempt + 1} failed for Block {idx + 1}: {e}")
                if attempt < max_retries - 1:
                    print(f"Retrying in {retry_delay} seconds...")
                    time.sleep(retry_delay)
                else:
                    print(f"Block {idx + 1} permanently failed after {max_retries} attempts. Skipping to next block.")
                    
        # Keep the script moving even if one block errors out completely
        if not success:
            with open(os.path.join(OUTPUT_DIR, "failed_blocks.log"), "a") as log:
                log.write(f"Block {idx + 1} failed with error.\n")

print("\nAutomated chunked audit finished successfully.")

Block 1: Text variation detected! Requesting localized analysis...

--- [DIFF REPORT FOR BLOCK 1] ---
This legislative comparison report analyzes the differences between the provided older revision and the newer revision of the **Mining Act 1978 (Western Australia)**.

---

### 1. Executive Summary
The changes between the two provided text segments are **minimal and highly targeted**. The amendments do not alter the fundamental structure of the Act but focus on two primary areas: the modernization of terminology regarding energy resources (specifically the inclusion of greenhouse gas storage) and the repeal of specific rights regarding oil shale and coal. There are no changes to penalties or administrative procedures in the segments provided.

---

### 2. Specific Section Amendments

| Section | Status | Change Description |
| :--- | :--- | :--- |
| **Section 8** | **Altered** | Updated definitions of "minerals" and references to petroleum legislation to include "Greenhouse Gas Storage

In [13]:
print(respone_text)


--- [DIFF REPORT FOR BLOCK 1] ---This legislative comparison report analyzes the differences between the provided older revision and the newer revision of the **Mining Act 1978 (Western Australia)**.

---

### 1. Executive Summary
The changes between the two provided text segments are **minimal and highly targeted**. The amendments do not alter the fundamental structure of the Act but focus on two primary areas: the modernization of terminology regarding energy resources (specifically the inclusion of greenhouse gas storage) and the repeal of specific rights regarding oil shale and coal. There are no changes to penalties or administrative procedures in the segments provided.

---

### 2. Specific Section Amendments

| Section | Status | Change Description |
| :--- | :--- | :--- |
| **Section 8** | **Altered** | Updated definitions of "minerals" and references to petroleum legislation to include "Greenhouse Gas Storage." |
| **Section 8A** | **Deleted** | The section "Rights in respect

In [14]:
master_prompt = f"""
You are an expert legal data analyst and legislative archivist. Your task is to compile a definitive, 
professional "Legislative Reform Audit Report" based on the provided block-by-block differences found in the Western Australian Mining Act 1978.

Please synthesize the provided findings into a single, comprehensive master report using the explicit structure below. 

---
### Strict Content Guidelines:
- Base the entire report ONLY on the provided diff reports. Do not invent, speculate, or infer information outside of this context.
- Maintain strict formatting accuracy. Avoid dense walls of text by using clear scannable lists, markdown tables, and bolding for key terms.
- CRITICAL: Remove all inline citations, meta-commentary, or bracketed source tags (e.g., do not write "[From CONTEXT...]" or "[cite: 1]"). The final output must be perfectly clean.
- Place the verbatim title of the legislation explicitly at the very start of the report.
---

[TITLE]
# Legislative Reform Audit Report: MINING ACT 1978

## 1. Executive Summary
Provide a high-level overview of the transition from the older revision (1 Sep 2025) to the newer revision (9 Sep 2025). 
Summarize the overarching themes of the updates, specifically highlighting the shift from a fragmented, 
passive system of individual lease conditions to a proactive, 
centralized regulatory framework for environmental oversight, approvals, and mine closure management.

## 2. Comprehensive Section-by-Section Amendments
Consolidate all section updates into an organized list sorted by the nature of the change:

### A. Major Part and Section Additions
Detail the newly introduced structures. Focus heavily on:
- **Part 4AA (Conditions and Approvals):** Break down its core Divisions (Divisions 1 through 7) and explain the newly established mechanisms like Eligible Mining Activities (EMA), 
EMA Notices, Excluded Area Notices, and Centralized Programmes of Work.
- **Sections 40A and 40B:** Introduce the definitions of "available land" and "conservation land" for miner's rights.
- **Sections 103AI through 103AV:** Detail the operational requirements for Programmes of Work, Mining Development and Closure Proposals, Mine Closure Plans, and Environmental Securities.

### B. Repealed and Deleted Provisions
List all standalone sections and subsections completely removed in the new revision, including:
- Section 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84, and 84AA.
- Explain what regulatory responsibilities (e.g., ground-disturbing equipment rules, injury to land reductions, and mine closure plan reviews) were stripped out of these specific lease clauses to be centralized elsewhere.

### C. Altered and Modernized Sections
Document existing sections that underwent structural or cross-referencing rewrites (e.g., Sections 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 140, and 162). 
Frame these updates around administrative simplification and updated cross-references matching Part 4AA.

## 3. Thematic and Textual Analysis

### A. Significant Shifts in Legal Terminology
Analyze specific phrasing updates that modernize the text or change its enforcement flexibility. Create a markdown table charting these specific shifts:
- The transition from "Deemed" to "Taken" (e.g., "is taken to be granted").
- The expansion from "Mining Proposal" to "Mining Development and Closure Proposal."
- The shift from "he deems" to "the Governor thinks" to ensure gender-neutral formal updates.

### B. Broadening of Liability and Safety Scope
Contrast the phrasing of the liability limitations (specifically referencing Section 20 and Section 40D). 
Detail how changing the narrow standard targeting "fire damage to trees" to a broader standard ("damage or injury to property or livestock whether resulting from fire... or any other cause") widens protections for land occupiers.

### C. Regulatory Restructuring of Ground Disturbance & Environment
Explain how environmental compliance moved from a fragmented system attached as ad-hoc conditions on specific tenements into a standardized, 
unified baseline via Part 4AA. 

### D. Compliance, Penalties, and Forfeiture Triggers
Detail how the legal triggers for forfeiture have expanded under Section 63A and Section 96 to encompass the new suite of Part 4AA compliance requirements. 
Highlight the operational mechanism of Section 103AV which binds environmental remediation directly to financial security and forfeiture.

## 4. Administrative and Historical Metadata
Summarize the structural changes to the Act's layout, specifically the transition to a comprehensive compilation format including chronological amendment tables (tracking amendments from 1987 to 2025), 
uncommenced provisions tables, reprint histories, and the relocation of transitional savings provisions to the "Other notes" section.

---
### SOURCE DATA FOR COMPARISON:
{respone_text}

it is important to respond in Markdown
"""

In [15]:
master_response = client.models.generate_content(model='gemini-3.1-flash-lite',
                                                    contents=master_prompt,
                                                        config=types.GenerateContentConfig(temperature=0.0)
                                                    )

In [16]:
display(Markdown(master_response.text))

# Legislative Reform Audit Report: MINING ACT 1978

## 1. Executive Summary
The transition from the 1 September 2025 revision to the 9 September 2025 revision represents a strategic shift toward a centralized, proactive regulatory framework for environmental oversight and mine closure. The amendments move the Act away from a fragmented system of ad-hoc lease conditions toward a standardized, unified baseline. Key themes include the formalization of environmental securities, the integration of greenhouse gas storage into the regulatory scope, and the establishment of a comprehensive administrative structure for mine development and closure.

## 2. Comprehensive Section-by-Section Amendments

### A. Major Part and Section Additions
*   **Part 4AA (Conditions and Approvals):** Introduces a centralized framework for environmental management.
    *   **Divisions 1–7:** Establishes the lifecycle of mining activities, including the classification of Eligible Mining Activities (EMA).
    *   **Mechanisms:** Implements EMA Notices, Excluded Area Notices, and Centralized Programmes of Work to replace fragmented tenement-specific conditions.
*   **Sections 40A and 40B:** Defines the scope of miner’s rights, specifically distinguishing between "available land" (Crown or conservation land not subject to a tenement) and "conservation land" (Class A reserves and land under the Conservation and Parks Commission).
*   **Sections 103AI through 103AV:** Codifies operational requirements for:
    *   Programmes of Work and Mining Development and Closure Proposals.
    *   Mandatory Mine Closure Plans.
    *   Environmental Securities, linking financial liability directly to remediation obligations.

### B. Repealed and Deleted Provisions
The following sections were repealed to eliminate redundant or fragmented regulatory clauses, centralizing these responsibilities under Part 4AA:
*   **Repealed Sections:** 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84, and 84AA.
*   **Regulatory Impact:** These deletions removed specific, isolated rules regarding ground-disturbing equipment, injury to land, and ad-hoc mine closure plan reviews, which are now governed by the standardized requirements of Part 4AA.

### C. Altered and Modernized Sections
Sections 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 140, and 162 underwent structural rewrites. These updates primarily serve to align existing administrative processes with the new Part 4AA cross-references and to modernize terminology regarding energy resources.

## 3. Thematic and Textual Analysis

### A. Significant Shifts in Legal Terminology

| Old Terminology | New Terminology | Context |
| :--- | :--- | :--- |
| "Deemed" | "Taken" | Used in granting processes (e.g., "is taken to be granted"). |
| "Mining Proposal" | "Mining Development and Closure Proposal" | Reflects the expanded scope of environmental planning. |
| "He deems" | "The Governor thinks" | Gender-neutral formalization of executive powers. |

### B. Broadening of Liability and Safety Scope
The Act has expanded the liability standard for land occupiers. Previously, protections were narrowly focused on "fire damage to trees." Under the updated Sections 20 and 40D, the scope is broadened to include "damage or injury to property or livestock whether resulting from fire... or any other cause," significantly increasing the protection afforded to landholders.

### C. Regulatory Restructuring of Ground Disturbance & Environment
Environmental compliance has transitioned from a system of ad-hoc conditions attached to individual tenements to a unified, baseline regulatory framework. By utilizing Part 4AA, the Act now mandates that all ground-disturbing activities adhere to centralized standards, ensuring consistency across all mining operations regardless of the specific tenement type.

### D. Compliance, Penalties, and Forfeiture Triggers
Forfeiture triggers under Sections 63A and 96 have been expanded to include non-compliance with the new Part 4AA requirements. Section 103AV serves as the primary enforcement mechanism, binding environmental remediation directly to financial security. Failure to meet these obligations now serves as a direct trigger for the forfeiture of the tenement.

## 4. Administrative and Historical Metadata
The Act has transitioned to a comprehensive compilation format. This includes:
*   **Chronological Amendment Tables:** Tracking legislative history from 1987 through 2025.
*   **Uncommenced Provisions:** A dedicated table for future-dated legislative changes.
*   **Reprint History:** A formalized record of all previous versions of the Act.
*   **Transitional Savings:** Relocation of transitional and savings provisions to the "Other notes" section to improve the readability of the primary legislative text.

In [17]:
new_date_tag = extract_date_from_filename(new_mining_act_file) 
old_date_tag = extract_date_from_filename(old_mining_act_file)

dynamic_filename = f"Legislative_Reform_Audit_Report_{old_date_tag}_to_{new_date_tag}.docx"

save_markdown_to_docx(master_response.text, dynamic_filename)

Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026.docx


In [18]:
def tranlate_master_respone(master_response_text,target_language):

    master_response_tranlate = f"""
                                You are an expert legal translator and legislative archivist fluent in both 
                                English and formal {target_language} legal terminology.
    
                                Your task is to translate the provided English "Legislative Reform Audit Report" 
                                into professional, high-level legal {target_language}.
                                
                                ---
                                ### Strict Content & Formatting Guidelines:
                                - Do not interpret, expand, or summarize the text. 
                                Translate the technical legal content exactly as written.
                                - Maintain the exact markdown layout, including headings (#, ##, ###), bullet lists, 
                                and markdown table structures.
                                - Keep proper nouns, citations, alphanumeric codes, 
                                and official statutory references (e.g., "Mining Act 1978", "Part 4AA", "Section 103AV") 
                                in their original format unless a direct, universally accepted legal equivalent exists in {target_language}.
                                - Ensure the final output is completely clean and formatted entirely in Markdown. 
                                Do not include any conversational introductions, meta-commentary, or closing remarks.
                                ---
                                
                                ### REPORT TO TRANSLATE:
                                {master_response_text}
                                """
    
    master_response_tranlate = client.models.generate_content(model='gemini-3.1-flash-lite',
                                                        contents=master_response_tranlate,
                                                            config=types.GenerateContentConfig(temperature=0.0)
                                                        )
    display(Markdown(master_response_tranlate.text))

    save_markdown_to_docx(master_response_tranlate.text, 
                          f"Legislative_Reform_Audit_Report_{old_date_tag}_to_{new_date_tag}_{target_language}.docx")

In [19]:
list_of_target_language = ['Spanish',
                          'Hindi',
                          'Tamil',
                          'French',
                          'German',
                            'Italian',
                           'Polish'
                          ]

for i in list_of_target_language:
    tranlate_master_respone(master_response,i)

# Informe de Auditoría de Reforma Legislativa: MINING ACT 1978

## 1. Resumen Ejecutivo
La transición de la revisión del 1 de septiembre de 2025 a la revisión del 9 de septiembre de 2025 representa un cambio estratégico hacia un marco regulatorio centralizado y proactivo para la supervisión ambiental y el cierre de minas. Las enmiendas alejan a la Ley de un sistema fragmentado de condiciones de arrendamiento *ad-hoc* hacia una base normativa estandarizada y unificada. Los temas clave incluyen la formalización de garantías ambientales, la integración del almacenamiento de gases de efecto invernadero en el ámbito regulatorio y el establecimiento de una estructura administrativa integral para el desarrollo y cierre de minas.

## 2. Enmiendas Integrales Sección por Sección

### A. Adiciones Importantes de Partes y Secciones
*   **Part 4AA (Conditions and Approvals):** Introduce un marco centralizado para la gestión ambiental.
    *   **Divisions 1–7:** Establece el ciclo de vida de las actividades mineras, incluida la clasificación de Actividades Mineras Elegibles (EMA, por sus siglas en inglés).
    *   **Mecanismos:** Implementa Notificaciones de EMA, Notificaciones de Áreas Excluidas y Programas de Trabajo Centralizados para reemplazar las condiciones fragmentadas específicas de cada concesión.
*   **Sections 40A y 40B:** Define el alcance de los derechos del minero, distinguiendo específicamente entre "tierras disponibles" (*available land* — tierras de la Corona o de conservación no sujetas a una concesión) y "tierras de conservación" (*conservation land* — reservas de Clase A y tierras bajo la Comisión de Conservación y Parques).
*   **Sections 103AI a 103AV:** Codifica los requisitos operativos para:
    *   Programas de Trabajo y Propuestas de Desarrollo y Cierre de Minas.
    *   Planes de Cierre de Minas Obligatorios.
    *   Garantías Ambientales, vinculando la responsabilidad financiera directamente a las obligaciones de remediación.

### B. Disposiciones Derogadas y Eliminadas
Las siguientes secciones fueron derogadas para eliminar cláusulas regulatorias redundantes o fragmentadas, centralizando estas responsabilidades bajo la Part 4AA:
*   **Secciones Derogadas:** 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84 y 84AA.
*   **Impacto Regulatorio:** Estas eliminaciones suprimieron reglas específicas y aisladas relativas a equipos que alteran el suelo, daños a la propiedad y revisiones de planes de cierre de minas *ad-hoc*, las cuales ahora se rigen por los requisitos estandarizados de la Part 4AA.

### C. Secciones Alteradas y Modernizadas
Las secciones 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 140 y 162 fueron objeto de reescrituras estructurales. Estas actualizaciones sirven principalmente para alinear los procesos administrativos existentes con las nuevas referencias cruzadas de la Part 4AA y para modernizar la terminología relativa a los recursos energéticos.

## 3. Análisis Temático y Textual

### A. Cambios Significativos en la Terminología Jurídica

| Terminología Anterior | Terminología Nueva | Contexto |
| :--- | :--- | :--- |
| "Deemed" | "Taken" | Utilizado en procesos de concesión (ej. "se considera otorgado"). |
| "Mining Proposal" | "Mining Development and Closure Proposal" | Refleja el alcance ampliado de la planificación ambiental. |
| "He deems" | "The Governor thinks" | Formalización neutral en cuanto al género de los poderes ejecutivos. |

### B. Ampliación del Alcance de Responsabilidad y Seguridad
La Ley ha ampliado el estándar de responsabilidad para los ocupantes de tierras. Anteriormente, las protecciones se centraban estrechamente en "daños por incendio a los árboles". Bajo las secciones 20 y 40D actualizadas, el alcance se amplía para incluir "daños o lesiones a la propiedad o al ganado, ya sea como resultado de un incendio... o cualquier otra causa", aumentando significativamente la protección otorgada a los propietarios de tierras.

### C. Reestructuración Regulatoria de la Alteración del Suelo y el Medio Ambiente
El cumplimiento ambiental ha pasado de un sistema de condiciones *ad-hoc* adjuntas a concesiones individuales a un marco regulatorio base unificado. Al utilizar la Part 4AA, la Ley ahora exige que todas las actividades que alteran el suelo se adhieran a estándares centralizados, garantizando la consistencia en todas las operaciones mineras independientemente del tipo de concesión específica.

### D. Cumplimiento, Sanciones y Disparadores de Caducidad
Los disparadores de caducidad (*forfeiture triggers*) bajo las secciones 63A y 96 se han ampliado para incluir el incumplimiento de los nuevos requisitos de la Part 4AA. La sección 103AV sirve como el mecanismo de ejecución principal, vinculando la remediación ambiental directamente a la garantía financiera. El incumplimiento de estas obligaciones ahora sirve como un disparador directo para la caducidad de la concesión.

## 4. Metadatos Administrativos e Históricos
La Ley ha hecho la transición a un formato de compilación integral. Esto incluye:
*   **Tablas Cronológicas de Enmiendas:** Seguimiento del historial legislativo desde 1987 hasta 2025.
*   **Disposiciones no iniciadas:** Una tabla dedicada a los cambios legislativos con fecha futura.
*   **Historial de Reimpresión:** Un registro formalizado de todas las versiones anteriores de la Ley.
*   **Ahorros Transitorios:** Reubicación de las disposiciones transitorias y de ahorro a la sección de "Otras notas" para mejorar la legibilidad del texto legislativo principal.

Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Spanish.docx


# विधायी सुधार ऑडिट रिपोर्ट: MINING ACT 1978

## 1. कार्यकारी सारांश
1 सितंबर 2025 के संशोधन से 9 सितंबर 2025 के संशोधन तक का संक्रमण, पर्यावरणीय निगरानी और खदान समापन के लिए एक केंद्रीकृत, सक्रिय विनियामक ढांचे की ओर एक रणनीतिक बदलाव का प्रतिनिधित्व करता है। ये संशोधन अधिनियम को तदर्थ (ad-hoc) पट्टा शर्तों की खंडित प्रणाली से दूर ले जाकर एक मानकीकृत, एकीकृत आधार रेखा की ओर ले जाते हैं। प्रमुख विषयों में पर्यावरणीय प्रतिभूतियों का औपचारिककरण, विनियामक दायरे में ग्रीनहाउस गैस भंडारण का एकीकरण, और खदान विकास तथा समापन के लिए एक व्यापक प्रशासनिक ढांचे की स्थापना शामिल है।

## 2. व्यापक धारा-वार संशोधन

### A. प्रमुख भाग और धाराएं जोड़ना
*   **Part 4AA (शर्तें और अनुमोदन):** पर्यावरणीय प्रबंधन के लिए एक केंद्रीकृत ढांचा प्रस्तुत करता है।
    *   **Divisions 1–7:** खनन गतिविधियों के जीवनचक्र को स्थापित करता है, जिसमें पात्र खनन गतिविधियों (Eligible Mining Activities - EMA) का वर्गीकरण शामिल है।
    *   **तंत्र:** खंडित पट्टा-विशिष्ट शर्तों को बदलने के लिए EMA नोटिस, बहिष्कृत क्षेत्र नोटिस (Excluded Area Notices), और कार्य के केंद्रीकृत कार्यक्रम (Centralized Programmes of Work) लागू करता है।
*   **Sections 40A और 40B:** खनिक के अधिकारों के दायरे को परिभाषित करता है, विशेष रूप से "उपलब्ध भूमि" (क्राउन या संरक्षण भूमि जो किसी पट्टे के अधीन नहीं है) और "संरक्षण भूमि" (क्लास A रिज़र्व और संरक्षण एवं पार्क आयोग के अधीन भूमि) के बीच अंतर स्पष्ट करता है।
*   **Sections 103AI से 103AV:** निम्नलिखित के लिए परिचालन आवश्यकताओं को संहिताबद्ध करता है:
    *   कार्य के कार्यक्रम और खनन विकास एवं समापन प्रस्ताव।
    *   अनिवार्य खदान समापन योजनाएं।
    *   पर्यावरणीय प्रतिभूतियां, जो वित्तीय दायित्व को सीधे उपचारात्मक दायित्वों से जोड़ती हैं।

### B. निरस्त और विलोपित प्रावधान
अनावश्यक या खंडित विनियामक खंडों को समाप्त करने और इन जिम्मेदारियों को Part 4AA के तहत केंद्रीकृत करने के लिए निम्नलिखित धाराओं को निरस्त किया गया:
*   **निरस्त धाराएं:** 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84, और 84AA।
*   **विनियामक प्रभाव:** इन विलोपनों ने भूमि को नुकसान पहुंचाने वाले उपकरणों, भूमि को क्षति, और तदर्थ खदान समापन योजना समीक्षाओं से संबंधित विशिष्ट, पृथक नियमों को हटा दिया है, जो अब Part 4AA की मानकीकृत आवश्यकताओं द्वारा शासित होते हैं।

### C. परिवर्तित और आधुनिक धाराएं
धारा 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 140, और 162 में संरचनात्मक पुनर्लेखन किया गया। ये अद्यतन मुख्य रूप से मौजूदा प्रशासनिक प्रक्रियाओं को नए Part 4AA क्रॉस-रेफरेंस के साथ संरेखित करने और ऊर्जा संसाधनों के संबंध में शब्दावली को आधुनिक बनाने के लिए हैं।

## 3. विषयगत और पाठ्य विश्लेषण

### A. कानूनी शब्दावली में महत्वपूर्ण बदलाव

| पुरानी शब्दावली | नई शब्दावली | संदर्भ |
| :--- | :--- | :--- |
| "Deemed" (समझा गया) | "Taken" (माना गया) | अनुदान प्रक्रियाओं में प्रयुक्त (जैसे, "अनुदानित माना जाता है")। |
| "Mining Proposal" | "Mining Development and Closure Proposal" | पर्यावरणीय योजना के विस्तारित दायरे को दर्शाता है। |
| "He deems" | "The Governor thinks" | कार्यकारी शक्तियों का लिंग-तटस्थ औपचारिककरण। |

### B. दायित्व और सुरक्षा के दायरे का विस्तार
अधिनियम ने भूमि कब्जाधारियों के लिए दायित्व मानक का विस्तार किया है। पहले, सुरक्षा केवल "पेड़ों को आग से होने वाले नुकसान" तक सीमित थी। अद्यतन धाराओं 20 और 40D के तहत, दायरे को "संपत्ति या पशुधन को नुकसान या चोट, चाहे वह आग से हो... या किसी अन्य कारण से" तक व्यापक बना दिया गया है, जिससे भूमिधारकों को मिलने वाली सुरक्षा में काफी वृद्धि हुई है।

### C. भूमि विक्षोभ और पर्यावरण का विनियामक पुनर्गठन
पर्यावरणीय अनुपालन व्यक्तिगत पट्टों से जुड़ी तदर्थ शर्तों की प्रणाली से हटकर एक एकीकृत, आधारभूत विनियामक ढांचे में परिवर्तित हो गया है। Part 4AA का उपयोग करके, अधिनियम अब यह अनिवार्य करता है कि सभी भूमि-विक्षोभ गतिविधियां केंद्रीकृत मानकों का पालन करें, जिससे पट्टे के प्रकार की परवाह किए बिना सभी खनन कार्यों में निरंतरता सुनिश्चित हो सके।

### D. अनुपालन, दंड, और जब्ती के ट्रिगर
धारा 63A और 96 के तहत जब्ती के ट्रिगर्स का विस्तार किया गया है ताकि नए Part 4AA आवश्यकताओं के गैर-अनुपालन को शामिल किया जा सके। धारा 103AV प्राथमिक प्रवर्तन तंत्र के रूप में कार्य करती है, जो पर्यावरणीय उपचार को सीधे वित्तीय सुरक्षा से जोड़ती है। इन दायित्वों को पूरा करने में विफलता अब पट्टे की जब्ती के लिए एक सीधा ट्रिगर है।

## 4. प्रशासनिक और ऐतिहासिक मेटाडेटा
अधिनियम एक व्यापक संकलन प्रारूप में परिवर्तित हो गया है। इसमें शामिल हैं:
*   **कालानुक्रमिक संशोधन तालिकाएं:** 1987 से 2025 तक के विधायी इतिहास को ट्रैक करना।
*   **अप्रवर्तित प्रावधान:** भविष्य की तारीख वाले विधायी परिवर्तनों के लिए एक समर्पित तालिका।
*   **पुनर्मुद्रण इतिहास:** अधिनियम के सभी पिछले संस्करणों का एक औपचारिक रिकॉर्ड।
*   **संक्रमणकालीन बचत:** प्राथमिक विधायी पाठ की पठनीयता में सुधार के लिए संक्रमणकालीन और बचत प्रावधानों को "अन्य नोट्स" अनुभाग में स्थानांतरित करना।

Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Hindi.docx


# சட்ட சீர்திருத்த தணிக்கை அறிக்கை: MINING ACT 1978

## 1. நிர்வாகச் சுருக்கம்
1 செப்டம்பர் 2025 திருத்தத்திலிருந்து 9 செப்டம்பர் 2025 திருத்தத்திற்கு மாறுவது, சுற்றுச்சூழல் மேற்பார்வை மற்றும் சுரங்க மூடல் ஆகியவற்றிற்கான மையப்படுத்தப்பட்ட, செயலூக்கமுள்ள ஒழுங்குமுறை கட்டமைப்பை நோக்கிய ஒரு மூலோபாய மாற்றத்தைக் குறிக்கிறது. இந்தத் திருத்தங்கள், சட்டத்தை தற்காலிக குத்தகை நிபந்தனைகளின் சிதறிய அமைப்பிலிருந்து, தரப்படுத்தப்பட்ட, ஒருங்கிணைந்த அடிப்படைக்கு மாற்றுகின்றன. சுற்றுச்சூழல் பிணையங்களை முறைப்படுத்துதல், பசுமை இல்ல வாயு சேமிப்பை ஒழுங்குமுறை வரம்பிற்குள் ஒருங்கிணைத்தல் மற்றும் சுரங்க மேம்பாடு மற்றும் மூடல் ஆகியவற்றிற்கான விரிவான நிர்வாக கட்டமைப்பை நிறுவுதல் ஆகியவை முக்கிய கருப்பொருள்களாகும்.

## 2. விரிவான பிரிவு வாரியான திருத்தங்கள்

### A. முக்கிய பகுதி மற்றும் பிரிவு சேர்த்தல்கள்
*   **Part 4AA (நிபந்தனைகள் மற்றும் ஒப்புதல்கள்):** சுற்றுச்சூழல் மேலாண்மைக்கான மையப்படுத்தப்பட்ட கட்டமைப்பை அறிமுகப்படுத்துகிறது.
    *   **Divisions 1–7:** தகுதியுள்ள சுரங்க நடவடிக்கைகள் (Eligible Mining Activities - EMA) வகைப்பாடு உட்பட, சுரங்க நடவடிக்கைகளின் வாழ்க்கைச் சுழற்சியை நிறுவுகிறது.
    *   **செயல்முறைகள்:** சிதறிய குத்தகை-குறிப்பிட்ட நிபந்தனைகளுக்குப் பதிலாக, EMA அறிவிப்புகள், விலக்கப்பட்ட பகுதி அறிவிப்புகள் மற்றும் மையப்படுத்தப்பட்ட பணித் திட்டங்களைச் செயல்படுத்துகிறது.
*   **Sections 40A மற்றும் 40B:** சுரங்கத் தொழிலாளியின் உரிமைகளின் வரம்பை வரையறுக்கிறது, குறிப்பாக "கிடைக்கக்கூடிய நிலம்" (குத்தகைக்கு உட்படுத்தப்படாத கிரீடம் அல்லது பாதுகாப்பு நிலம்) மற்றும் "பாதுகாப்பு நிலம்" (வகுப்பு A இருப்புக்கள் மற்றும் பாதுகாப்பு மற்றும் பூங்காக்கள் ஆணையத்தின் கீழ் உள்ள நிலம்) ஆகியவற்றுக்கு இடையே வேறுபடுத்துகிறது.
*   **Sections 103AI முதல் 103AV வரை:** பின்வருவனவற்றிற்கான செயல்பாட்டுத் தேவைகளை முறைப்படுத்துகிறது:
    *   பணித் திட்டங்கள் மற்றும் சுரங்க மேம்பாடு மற்றும் மூடல் முன்மொழிவுகள்.
    *   கட்டாய சுரங்க மூடல் திட்டங்கள்.
    *   சுற்றுச்சூழல் பிணையங்கள், நிதிப் பொறுப்பை நேரடியாக மறுசீரமைப்பு கடமைகளுடன் இணைத்தல்.

### B. நீக்கப்பட்ட மற்றும் ரத்து செய்யப்பட்ட விதிகள்
ஒழுங்குமுறை உட்பிரிவுகளில் உள்ள தேவையற்ற அல்லது சிதறிய பகுதிகளை நீக்கவும், இந்தப் பொறுப்புகளை Part 4AA-ன் கீழ் மையப்படுத்தவும் பின்வரும் பிரிவுகள் ரத்து செய்யப்பட்டன:
*   **ரத்து செய்யப்பட்ட பிரிவுகள்:** 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84, மற்றும் 84AA.
*   **ஒழுங்குமுறை தாக்கம்:** இந்த நீக்கங்கள், தரை-சீர்குலைக்கும் உபகரணங்கள், நிலத்திற்கு ஏற்படும் பாதிப்பு மற்றும் தற்காலிக சுரங்க மூடல் திட்ட மறுஆய்வுகள் தொடர்பான குறிப்பிட்ட, தனிமைப்படுத்தப்பட்ட விதிகளை நீக்கியுள்ளன, இவை இப்போது Part 4AA-ன் தரப்படுத்தப்பட்ட தேவைகளால் நிர்வகிக்கப்படுகின்றன.

### C. மாற்றப்பட்ட மற்றும் நவீனப்படுத்தப்பட்ட பிரிவுகள்
பிரிவுகள் 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 140, மற்றும் 162 ஆகியவை கட்டமைப்பு ரீதியாக மீண்டும் எழுதப்பட்டன. இந்த புதுப்பிப்புகள் முதன்மையாக ஏற்கனவே உள்ள நிர்வாக செயல்முறைகளை புதிய Part 4AA குறுக்கு-குறிப்புகளுடன் சீரமைக்கவும், எரிசக்தி வளங்கள் தொடர்பான கலைச்சொற்களை நவீனப்படுத்தவும் உதவுகின்றன.

## 3. கருப்பொருள் மற்றும் உரை பகுப்பாய்வு

### A. சட்டக் கலைச்சொற்களில் குறிப்பிடத்தக்க மாற்றங்கள்

| பழைய கலைச்சொல் | புதிய கலைச்சொல் | சூழல் |
| :--- | :--- | :--- |
| "Deemed" (கருதப்படுவது) | "Taken" (எடுத்துக்கொள்ளப்படுவது) | அனுமதி வழங்கும் செயல்முறைகளில் பயன்படுத்தப்படுகிறது (எ.கா., "அனுமதி வழங்கப்பட்டதாகக் கருதப்படுகிறது"). |
| "Mining Proposal" | "Mining Development and Closure Proposal" | சுற்றுச்சூழல் திட்டமிடலின் விரிவான வரம்பைப் பிரதிபலிக்கிறது. |
| "He deems" | "The Governor thinks" | நிர்வாக அதிகாரங்களின் பாலின-நடுநிலை முறைப்படுத்தல். |

### B. பொறுப்பு மற்றும் பாதுகாப்பு வரம்பை விரிவுபடுத்துதல்
இந்தச் சட்டம் நில ஆக்கிரமிப்பாளர்களுக்கான பொறுப்புத் தரத்தை விரிவுபடுத்தியுள்ளது. முன்பு, பாதுகாப்புகள் "மரங்களுக்கு ஏற்படும் தீ சேதம்" என்பதில் மட்டுமே குறுகிய கவனம் செலுத்தின. புதுப்பிக்கப்பட்ட பிரிவுகள் 20 மற்றும் 40D-ன் கீழ், வரம்பானது "தீயினால் அல்லது வேறு ஏதேனும் காரணத்தினால் சொத்து அல்லது கால்நடைகளுக்கு ஏற்படும் சேதம் அல்லது காயம்" என விரிவுபடுத்தப்பட்டுள்ளது, இது நில உரிமையாளர்களுக்கு வழங்கப்படும் பாதுகாப்பை கணிசமாக அதிகரிக்கிறது.

### C. தரை சீர்குலைவு மற்றும் சுற்றுச்சூழலின் ஒழுங்குமுறை மறுசீரமைப்பு
சுற்றுச்சூழல் இணக்கமானது, தனிப்பட்ட குத்தகைகளுடன் இணைக்கப்பட்ட தற்காலிக நிபந்தனைகளின் அமைப்பிலிருந்து, ஒருங்கிணைந்த, அடிப்படை ஒழுங்குமுறை கட்டமைப்பிற்கு மாறியுள்ளது. Part 4AA-ஐப் பயன்படுத்துவதன் மூலம், அனைத்து தரை-சீர்குலைக்கும் நடவடிக்கைகளும் மையப்படுத்தப்பட்ட தரநிலைகளுக்கு இணங்க வேண்டும் என்று சட்டம் இப்போது கட்டாயமாக்குகிறது, இது குறிப்பிட்ட குத்தகை வகையைப் பொருட்படுத்தாமல் அனைத்து சுரங்க நடவடிக்கைகளிலும் நிலைத்தன்மையை உறுதி செய்கிறது.

### D. இணக்கம், அபராதங்கள் மற்றும் பறிமுதல் தூண்டுதல்கள்
பிரிவுகள் 63A மற்றும் 96-ன் கீழ் பறிமுதல் தூண்டுதல்கள், புதிய Part 4AA தேவைகளுக்கு இணங்காததையும் உள்ளடக்கும் வகையில் விரிவுபடுத்தப்பட்டுள்ளன. பிரிவு 103AV முதன்மை அமலாக்க பொறிமுறையாக செயல்படுகிறது, இது சுற்றுச்சூழல் மறுசீரமைப்பை நேரடியாக நிதிப் பாதுகாப்புடன் பிணைக்கிறது. இந்த கடமைகளை நிறைவேற்றத் தவறுவது இப்போது குத்தகையைப் பறிமுதல் செய்வதற்கான நேரடித் தூண்டுதலாகச் செயல்படுகிறது.

## 4. நிர்வாக மற்றும் வரலாற்று மெட்டாடேட்டா
இந்தச் சட்டம் ஒரு விரிவான தொகுப்பு வடிவத்திற்கு மாறியுள்ளது. இதில் பின்வருவன அடங்கும்:
*   **காலவரிசை திருத்த அட்டவணைகள்:** 1987 முதல் 2025 வரையிலான சட்ட வரலாற்றைக் கண்காணித்தல்.
*   **செயல்படுத்தப்படாத விதிகள்:** எதிர்கால தேதியிட்ட சட்ட மாற்றங்களுக்கான பிரத்யேக அட்டவணை.
*   **மறுபதிப்பு வரலாறு:** சட்டத்தின் அனைத்து முந்தைய பதிப்புகளின் முறைப்படுத்தப்பட்ட பதிவு.
*   **இடைக்கால சேமிப்புகள்:** முதன்மை சட்ட உரையின் வாசிப்புத்திறனை மேம்படுத்த, இடைக்கால மற்றும் சேமிப்பு விதிகளை "பிற குறிப்புகள்" (Other notes) பகுதிக்கு இடமாற்றம் செய்தல்.

...

Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_German.docx


# Rapporto di Revisione della Riforma Legislativa: MINING ACT 1978

## 1. Sintesi Esecutiva
La transizione dalla revisione del 1° settembre 2025 alla revisione del 9 settembre 2025 rappresenta un mutamento strategico verso un quadro normativo centralizzato e proattivo per la supervisione ambientale e la chiusura delle miniere. Gli emendamenti allontanano il *Mining Act* da un sistema frammentato di condizioni di locazione *ad hoc* verso una base normativa standardizzata e unificata. I temi chiave includono la formalizzazione delle garanzie ambientali, l'integrazione dello stoccaggio dei gas serra nell'ambito normativo e l'istituzione di una struttura amministrativa completa per lo sviluppo e la chiusura delle miniere.

## 2. Emendamenti Completi Sezione per Sezione

### A. Aggiunte Principali di Parti e Sezioni
*   **Part 4AA (Condizioni e Approvazioni):** Introduce un quadro centralizzato per la gestione ambientale.
    *   **Divisions 1–7:** Stabilisce il ciclo di vita delle attività minerarie, inclusa la classificazione delle Attività Minerarie Ammissibili (*Eligible Mining Activities* - EMA).
    *   **Meccanismi:** Implementa gli Avvisi EMA, gli Avvisi di Area Esclusa e i Programmi di Lavoro Centralizzati per sostituire le condizioni frammentate specifiche per ogni concessione (*tenement*).
*   **Sections 40A e 40B:** Definisce l'ambito dei diritti del minatore, distinguendo specificamente tra "terreno disponibile" (*available land* - terreno della Corona o terreno protetto non soggetto a concessione) e "terreno di conservazione" (*conservation land* - riserve di Classe A e terreni sotto la *Conservation and Parks Commission*).
*   **Sections 103AI fino a 103AV:** Codifica i requisiti operativi per:
    *   Programmi di Lavoro e Proposte di Sviluppo e Chiusura Mineraria.
    *   Piani di Chiusura Miniera Obbligatori.
    *   Garanzie Ambientali, collegando la responsabilità finanziaria direttamente agli obblighi di bonifica.

### B. Disposizioni Abrogate e Soppresse
Le seguenti sezioni sono state abrogate per eliminare clausole normative ridondanti o frammentate, centralizzando tali responsabilità sotto la Part 4AA:
*   **Sezioni Abrogate:** 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84 e 84AA.
*   **Impatto Normativo:** Tali soppressioni hanno rimosso norme specifiche e isolate riguardanti le attrezzature che alterano il suolo, i danni al terreno e le revisioni *ad hoc* dei piani di chiusura miniera, che sono ora disciplinate dai requisiti standardizzati della Part 4AA.

### C. Sezioni Modificate e Modernizzate
Le sezioni 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 140 e 162 sono state oggetto di riscritture strutturali. Tali aggiornamenti servono principalmente ad allineare i processi amministrativi esistenti con i nuovi rinvii della Part 4AA e a modernizzare la terminologia relativa alle risorse energetiche.

## 3. Analisi Tematica e Testuale

### A. Significativi Mutamenti nella Terminologia Giuridica

| Terminologia Precedente | Nuova Terminologia | Contesto |
| :--- | :--- | :--- |
| "Deemed" (Ritenuto) | "Taken" (Considerato) | Utilizzato nei processi di concessione (es. "is taken to be granted"). |
| "Mining Proposal" | "Mining Development and Closure Proposal" | Riflette l'ambito esteso della pianificazione ambientale. |
| "He deems" (Egli ritiene) | "The Governor thinks" (Il Governatore ritiene) | Formalizzazione neutra rispetto al genere dei poteri esecutivi. |

### B. Ampliamento dell'Ambito di Responsabilità e Sicurezza
L'Atto ha ampliato lo standard di responsabilità per gli occupanti del terreno. In precedenza, le tutele erano limitate ai "danni da incendio agli alberi". Ai sensi delle aggiornate Sezioni 20 e 40D, l'ambito è esteso per includere "danni o lesioni a proprietà o bestiame, derivanti da incendio... o da qualsiasi altra causa", aumentando significativamente la protezione accordata ai proprietari terrieri.

### C. Ristrutturazione Normativa dell'Alterazione del Suolo e dell'Ambiente
La conformità ambientale è passata da un sistema di condizioni *ad hoc* allegate alle singole concessioni a un quadro normativo di base unificato. Utilizzando la Part 4AA, l'Atto ora impone che tutte le attività che alterano il suolo aderiscano a standard centralizzati, garantendo coerenza in tutte le operazioni minerarie indipendentemente dal tipo specifico di concessione.

### D. Conformità, Sanzioni e Cause di Decadenza
Le cause di decadenza (*forfeiture*) ai sensi delle Sezioni 63A e 96 sono state ampliate per includere la non conformità con i nuovi requisiti della Part 4AA. La Sezione 103AV funge da principale meccanismo di applicazione, vincolando la bonifica ambientale direttamente alla garanzia finanziaria. L'inadempimento di tali obblighi costituisce ora una causa diretta per la decadenza della concessione.

## 4. Metadati Amministrativi e Storici
L'Atto è passato a un formato di compilazione completo. Ciò include:
*   **Tabelle Cronologiche degli Emendamenti:** Tracciamento della storia legislativa dal 1987 al 2025.
*   **Disposizioni non ancora in vigore:** Una tabella dedicata alle modifiche legislative con data futura.
*   **Cronologia delle Ristampe:** Un registro formalizzato di tutte le versioni precedenti dell'Atto.
*   **Disposizioni Transitorie e di Salvaguardia:** Ricollocazione delle disposizioni transitorie e di salvaguardia nella sezione "Altre note" per migliorare la leggibilità del testo legislativo primario.

Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Italian.docx


# Raport z audytu reformy legislacyjnej: MINING ACT 1978

## 1. Podsumowanie wykonawcze
Przejście od wersji ustawy z dnia 1 września 2025 r. do wersji z dnia 9 września 2025 r. stanowi strategiczną zmianę w kierunku scentralizowanych, proaktywnych ram regulacyjnych w zakresie nadzoru środowiskowego oraz likwidacji kopalń. Nowelizacja odchodzi od rozproszonego systemu doraźnych warunków dzierżawy na rzecz ujednoliconej, standardowej bazy regulacyjnej. Kluczowe zagadnienia obejmują formalizację zabezpieczeń środowiskowych, włączenie składowania gazów cieplarnianych w zakres regulacji oraz ustanowienie kompleksowej struktury administracyjnej dla rozwoju i zamykania kopalń.

## 2. Szczegółowy przegląd nowelizacji artykułów

### A. Główne dodatki do części i artykułów
*   **Part 4AA (Warunki i zatwierdzenia):** Wprowadza scentralizowane ramy zarządzania środowiskowego.
    *   **Divisions 1–7:** Ustanawiają cykl życia działalności górniczej, w tym klasyfikację kwalifikowalnej działalności górniczej (Eligible Mining Activities – EMA).
    *   **Mechanizmy:** Wdrażają zawiadomienia EMA (EMA Notices), zawiadomienia o wyłączonych obszarach (Excluded Area Notices) oraz scentralizowane programy prac (Centralized Programmes of Work) w celu zastąpienia rozproszonych warunków przypisanych do poszczególnych tytułów prawnych (tenement-specific conditions).
*   **Sections 40A i 40B:** Definiują zakres praw górnika, w szczególności rozróżniając „dostępny grunt” (grunt państwowy lub chroniony niepodlegający tytułowi prawnemu) oraz „grunt chroniony” (rezerwaty klasy A oraz grunty podlegające Conservation and Parks Commission).
*   **Sections 103AI do 103AV:** Kodyfikują wymogi operacyjne dotyczące:
    *   Programów prac oraz wniosków dotyczących rozwoju i zamykania kopalń (Mining Development and Closure Proposals).
    *   Obowiązkowych planów zamknięcia kopalń (Mine Closure Plans).
    *   Zabezpieczeń środowiskowych, wiążących odpowiedzialność finansową bezpośrednio z obowiązkami rekultywacyjnymi.

### B. Przepisy uchylone i usunięte
Poniższe artykuły zostały uchylone w celu wyeliminowania zbędnych lub rozproszonych klauzul regulacyjnych, poprzez scentralizowanie tych obowiązków w ramach Part 4AA:
*   **Uchylone artykuły:** 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84 oraz 84AA.
*   **Wpływ regulacyjny:** Usunięcia te wyeliminowały szczegółowe, odizolowane zasady dotyczące sprzętu naruszającego grunt, szkód w mieniu oraz doraźnych przeglądów planów zamknięcia kopalń, które obecnie podlegają ujednoliconym wymogom Part 4AA.

### C. Zmienione i zmodernizowane artykuły
Artykuły 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 140 oraz 162 zostały poddane zmianom strukturalnym. Aktualizacje te służą przede wszystkim dostosowaniu istniejących procesów administracyjnych do nowych odniesień w Part 4AA oraz modernizacji terminologii dotyczącej zasobów energetycznych.

## 3. Analiza tematyczna i tekstowa

### A. Istotne zmiany w terminologii prawnej

| Stara terminologia | Nowa terminologia | Kontekst |
| :--- | :--- | :--- |
| "Deemed" (Uznaje się) | "Taken" (Przyjmuje się) | Stosowane w procesach przyznawania (np. "is taken to be granted"). |
| "Mining Proposal" | "Mining Development and Closure Proposal" | Odzwierciedla rozszerzony zakres planowania środowiskowego. |
| "He deems" | "The Governor thinks" | Neutralna płciowo formalizacja uprawnień wykonawczych. |

### B. Rozszerzenie zakresu odpowiedzialności i bezpieczeństwa
Ustawa rozszerzyła standard odpowiedzialności wobec użytkowników gruntów. Wcześniej ochrona była wąsko skoncentrowana na "szkodach wyrządzonych przez ogień drzewom". Zgodnie ze zaktualizowanymi artykułami 20 i 40D, zakres ten został rozszerzony o "szkody lub uszczerbek na mieniu lub inwentarzu żywym, wynikające z ognia... lub jakiejkolwiek innej przyczyny", co znacząco zwiększa ochronę przysługującą właścicielom gruntów.

### C. Restrukturyzacja regulacyjna naruszania gruntu i środowiska
Zgodność ze środowiskiem przeszła z systemu doraźnych warunków przypisanych do poszczególnych tytułów prawnych na ujednolicone, podstawowe ramy regulacyjne. Dzięki wykorzystaniu Part 4AA, ustawa nakłada obecnie wymóg, aby wszelkie działania naruszające grunt były zgodne ze scentralizowanymi standardami, zapewniając spójność we wszystkich operacjach górniczych, niezależnie od rodzaju tytułu prawnego.

### D. Zgodność, kary i przesłanki przepadku
Przesłanki przepadku (forfeiture) na mocy artykułów 63A i 96 zostały rozszerzone o przypadki nieprzestrzegania nowych wymogów Part 4AA. Artykuł 103AV służy jako główny mechanizm egzekwowania, wiążący rekultywację środowiskową bezpośrednio z zabezpieczeniem finansowym. Niewypełnienie tych obowiązków stanowi obecnie bezpośrednią przesłankę przepadku tytułu prawnego.

## 4. Metadane administracyjne i historyczne
Ustawa przeszła na format kompleksowej kompilacji. Obejmuje to:
*   **Chronologiczne tabele nowelizacji:** Śledzące historię legislacyjną od 1987 do 2025 roku.
*   **Przepisy niewchodzące w życie:** Dedykowana tabela dla zmian legislacyjnych z odroczonym terminem wejścia w życie.
*   **Historia wydań (Reprint History):** Sformalizowany rejestr wszystkich poprzednich wersji ustawy.
*   **Przepisy przejściowe:** Przeniesienie przepisów przejściowych i końcowych do sekcji "Inne uwagi" (Other notes) w celu poprawy czytelności głównego tekstu legislacyjnego.

Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Polish.docx
